# 🔄 Actualizar Dashboard Eco Go — Completo
Corre el refresh de **todas** las secciones, tanto del dashboard interno (`index.html`)
como del de clientes (`index-clientes.html`). Los dos leen los mismos archivos de datos,
asi que con correr este notebook alcanza: no hace falta `refresh.bat` ni ningun otro aparte.

| Seccion | Fuente |
|---|---|
| Series Largas | `Anexo.xlsx` — anexo historico |
| Actividad IPI | `IPI - Todos.xlsx` |
| Precios (IPC + RPM) | `IPC TODESCA.xlsx` · `Graficos de dispersion.xlsx` · `Cuadro Mensual - Capitulos Nuevo.xlsx` |
| EMAE (Actividad) | `BD\Actividad\EMAE.xlsx` |
| Empleo | `Empleo_nuevo.xlsx` |
| Salarios | `Salarios.xlsx` |
| Tipo de Cambio | `TCR bandas.xlsx` · `Rofex.xlsx` · `com3500.xls` · `Copia de Blue.xlsx` |
| Reservas | `pasivos reservas.xlsx` · `Reservas brutas y depositos.xlsx` |
| Monetarias | `Copia de Agregados monetarios.xlsx` · `Monitor monetario mensual.xlsx` |
| Deuda | `Deuda Lopez Murphy.xlsx` · `Deuda en pesos.xlsx` · `Ejercicio refinanciamiento.xlsx` |
| Comercio exterior | `Comercio exterior.xlsx` · `Terminos del Intercambio.xlsx` · `Estimacion Comercio Ext.xlsx` |
| Internacional | `BD\Internacional\Monitor mundial\data\monitor-data.js` |
| Internacional · Consensus | el PDF de LatinFocus mas nuevo de `07 Tableros\Proyecciones internacionales` |
| Mercados | API EcoGo Markets |

> **RIGI** no se actualiza desde Excel: los datos estan embebidos en `rigi-clientes.html`.
>
> Si alguna serie queda vieja, la celda lo avisa con un `[WARN] ... meses de atraso`.
> Vale la pena leer los WARN del resumen: un dato que se deja de actualizar no rompe nada,
> el refresh termina bien igual.


In [1]:
import sys, os

DASHBOARD_DIR = r"C:\Users\fscalise\OneDrive - ECOGO S.A\BD\07 Tableros\EcoGo-Dashboard"
sys.path.insert(0, DASHBOARD_DIR)

# Importar funciones del refresh
import refresh as R

print(f"Dashboard: {DASHBOARD_DIR}")
print(f"Excel base: {R.BASE_EXCEL}")
print(f"Data dir:   {R.DATA_DIR}")

Dashboard: C:\Users\fscalise\OneDrive - ECOGO S.A\BD\07 Tableros\EcoGo-Dashboard
Excel base: C:\Users\fscalise\OneDrive - ECOGO S.A\BD
Data dir:   c:\Users\hzabaleta\OneDrive - ECOGO S.A\BD\07 Tableros\EcoGo-Dashboard\assets\data


## 0 · Series Largas (Anexo histórico)


In [ ]:
from extract_series_largas import run_extraction as run_series_largas
anexo_path = os.path.join(R.BASE_EXCEL, "03 Informes y Anexos", "Cuadros y Anexos", "Anexos nuevos", "Anexo.xlsx")
print(f"  Excel: {anexo_path}")
result = run_series_largas(anexo_path, DASHBOARD_DIR)
if result["ok"]:
    print(f"✅ Series Largas — {result['msg']}")
else:
    print(f"❌ Series Largas — {result['msg']}")

## 0b · Indicadores de Actividad (IPI)
`IPI - Todos.xlsx` (hoja `Hoja1`) — EMAE, IPI Manufacturero, ISAC, IPI Minero, ISSP.

In [ ]:
from extract_actividad_ipi import run_extraction as run_ipi
ipi_path = os.path.join(R.BASE_EXCEL, "Actividad", "IPI - Todos.xlsx")
result = run_ipi(ipi_path, DASHBOARD_DIR)
if result["ok"]:
    print(f"✅ Actividad IPI — {result['msg']}")
else:
    print(f"❌ Actividad IPI — {result['msg']}")

## 1 · Precios (IPC)


In [2]:
status = R.Status()
d = R.extract_precios(status)
if d:
    sz = R.save_data('precios', d)
    print(f"✅ precios.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

  [FAIL] Precios — No se encontró: C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Precios\IPC TODESCA.xlsx
  [FAIL] Precios: No se encontró: C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Precios\IPC TODESCA.xlsx


## 2 · EMAE (Actividad)


In [3]:
status = R.Status()
d = R.extract_emae_series(status)
if d:
    sz = R.save_emae_series(d)
    print(f"✅ emae_series.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

  [---]  EMAE Series — no se encontró C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Actividad\02 Indicador de Actividad CN2004\Base EsAE.xlsx
  [WARN] EMAE Series: no se encontró C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Actividad\02 Indicador de Actividad CN2004\Base EsAE.xlsx


## 2b · Monitor de Actividad
> Los tres cuadros de `Monitor de Actividad.xlsx` (`Cuadro Monitor`, `(var%)` y `(var%) (2)`).
> En el dashboard se ven como un solo cuadro con un botón para pasar de índice a var. % mensual o interanual.
> Los meses salen de la fila 2 del Excel: la ventana es móvil, cada mes nuevo agrega una columna.


In [ ]:
status = R.Status()
d = R.extract_monitor_actividad(status)
if d:
    print(f"✅ monitor_actividad.js — {R.save_data('monitor_actividad', d):,} bytes")
    print(f"   {len(d['series'])} series · {d['meses'][0]} a {d['meses'][-1]} · vistas: "
          + ", ".join(v["label"] for v in d["vistas"].values()))
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")


## 3 · Empleo

In [4]:
status = R.Status()
d = R.extract_empleo(status)
if d:
    sz = R.save_data('empleo', d)
    print(f"✅ empleo.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

  [FAIL] Empleo — No se encontró: C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Empleo\Empleo_nuevo.xlsx
  [FAIL] Empleo: No se encontró: C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Empleo\Empleo_nuevo.xlsx


## 4 · Salarios

In [5]:
status = R.Status()
d = R.extract_salarios(status)
if d:
    sz = R.save_data('salarios', d)
    print(f"✅ salarios.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

  [FAIL] Salarios — No se encontró: C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Empleo\Salarios.xlsx
  [FAIL] Salarios: No se encontró: C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Empleo\Salarios.xlsx


## 5 · Tipo de Cambio

In [6]:
status = R.Status()
d = R.extract_tipo_cambio(status)
if d:
    sz = R.save_data('tipo-cambio', d)
    print(f"✅ tipo-cambio.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

  [FAIL] Rofex — No se encontró: C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Tipo de Cambio\Rofex.xlsx
  [FAIL] Rofex: No se encontró: C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Tipo de Cambio\Rofex.xlsx


## 6 · Reservas

In [7]:
status = R.Status()
d = R.extract_reservas(status)
if d:
    sz = R.save_data('reservas', d)
    print(f"✅ reservas.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

  [---]  Reservas - RIN — no se encontro C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Monetarias\pasivos reservas.xlsx
  [---]  Reservas - G5 — no se encontro C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Monetarias\Reservas brutas y depósitos.xlsx
  [WARN] Reservas - RIN: no se encontro C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Monetarias\pasivos reservas.xlsx
  [WARN] Reservas - G5: no se encontro C:\Users\fscalise\OneDrive - ECOGO S.A\BD\Monetarias\Reservas brutas y depósitos.xlsx


## 6b · Monetarias
Agregados monetarios, préstamos privados y monetización de la economía — `Copia de Agregados monetarios.xlsx` y `Monitor monetario mensual.xlsx` (carpeta `BD/Monetarias`).

In [ ]:
status = R.Status()
d = R.extract_monetarias(status)
if d:
    sz = R.save_data('monetarias', d)
    print(f"✅ monetarias.js — {sz:,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")

## 6c · Deuda
> Cuadro Lopez Murphy, vencimientos en pesos y perfil de refinanciamiento.


In [ ]:
status = R.Status()
d = R.extract_deuda(status)
if d:
    print(f"✅ deuda.js — {R.save_data('deuda', d):,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")


## 6d · Comercio exterior
> Impo desestacionalizadas, saldos comerciales, términos de intercambio y cuadro mensual.


In [ ]:
status = R.Status()
d = R.extract_comercio(status)
if d:
    print(f"✅ comercio.js — {R.save_data('comercio', d):,} bytes")
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")


## 8 · Internacional (Monitor Mundial + LatinFocus)
> El Monitor mundial sale de `BD\Internacional\Monitor mundial\data\monitor-data.js`.
> LatinFocus toma solo el PDF más nuevo de `07 Tableros\Proyecciones internacionales`: alcanza con dejar el PDF del mes ahí.


In [ ]:
status = R.Status()
R.extract_internacional(status)
R.run_latinfocus(status)
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")


## 7 · Mercados (API)
> Si la API de origen viene con datos viejos, avisa con un `[WARN]`: el refresh puede estar OK y aun así publicar algo desactualizado.


In [ ]:
status = R.Status()
R.extract_mercados(status)
for r in status.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")


## ✅ Resumen final

In [10]:
from datetime import datetime
import os

data_dir = R.DATA_DIR
archivos = ['precios.js', 'emae_series.js', 'empleo.js', 'salarios.js',
            'tipo-cambio.js', 'reservas.js', 'monetarias.js', 'deuda.js',
            'comercio.js', 'internacional.js', 'internacional2.js',
            'mercados.js', 'series-largas.js', 'actividad_ipi.js']
print(f"{'Archivo':<22} {'Tamaño':>12} {'Modificado'}")
print('-' * 55)
for f in archivos:
    p = os.path.join(data_dir, f)
    if os.path.exists(p):
        sz = os.path.getsize(p)
        mt = datetime.fromtimestamp(os.path.getmtime(p)).strftime('%d/%m %H:%M')
        print(f"{f:<22} {sz:>10,} b  {mt}")
    else:
        print(f"{f:<22} {'—':>12}")
print()
print('Listo. Recargá el dashboard en el navegador.')


Archivo                      Tamaño Modificado
-------------------------------------------------------
precios.js                 62,846 b  08/07 17:18
emae_series.js            108,319 b  27/05 15:15
empleo.js                  31,644 b  08/07 17:16
salarios.js                31,488 b  08/07 17:17
tipo-cambio.js            916,241 b  08/07 17:17
reservas.js               376,609 b  08/07 17:18
internacional.js          756,425 b  08/07 17:17
mercados.js               267,368 b  17/07 16:04

Listo. Recargá el dashboard en el navegador.


## 🚀 Subir a GitHub
Copia los `.js`/`.json` actualizados al repo clonado y hace commit + push automático.

In [ ]:
print("\nSubiendo cambios a GitHub...")
status_git = R.Status()
R.push_to_github(status_git)
for r in status_git.results:
    print(f"  [{r[0]}] {r[1]}: {r[2]}")